In [ ]:
# ============================================================================
# INGENIERÍA DE CARACTERÍSTICAS
# ============================================================================

print("\n" + "="*70)
print("INGENIERÍA DE CARACTERÍSTICAS (FEATURE ENGINEERING)")
print("="*70)

df_features = df_merged.copy()

# 1. Calcular eGFR (CKD-EPI 2009)
print(f"\n[1] Calculando eGFR (CKD-EPI 2009)...")

def calculate_egfr(row):
    """Calcular eGFR usando ecuación CKD-EPI 2009"""
    scr = row['LBXSCR']  # Creatinina sérica en mg/dL
    age = row['RIDAGEYR']
    gender = row['RIAGENDR']  # 1=M, 2=F
    
    if pd.isna(scr) or pd.isna(age) or pd.isna(gender):
        return np.nan
    
    is_female = gender == 2
    kappa = 0.7 if is_female else 0.9
    alpha = -0.329 if is_female else -0.411
    
    ratio = scr / kappa
    
    egfr = (
        141 
        * np.power(min(ratio, 1), alpha) 
        * np.power(max(ratio, 1), -1.209) 
        * np.power(0.993, age)
    )
    
    return egfr

df_features['egfr'] = df_features.apply(calculate_egfr, axis=1)
print(f"  ✓ eGFR calculado: media={df_features['egfr'].mean():.1f}, rango=[{df_features['egfr'].min():.1f}, {df_features['egfr'].max():.1f}]")

# 2. Calcular MAP (Presión Arterial Media)
print(f"\n[2] Calculando MAP (Presión Arterial Media)...")
if 'BPXDI1' in df_features.columns:
    df_features['map'] = (df_features['BPXSY1'] + 2 * df_features['BPXDI1']) / 3
    print(f"  ✓ MAP calculado: media={df_features['map'].mean():.1f}, rango=[{df_features['map'].min():.1f}, {df_features['map'].max():.1f}]")
else:
    df_features['map'] = np.nan
    print(f"  ⚠ Presión diastólica no disponible")

# 3. Crear indicador de género (Female)
print(f"\n[3] Creando indicador de género...")
df_features['is_female'] = (df_features['RIAGENDR'] == 2).astype(int)
print(f"  ✓ Mujeres: {df_features['is_female'].sum()} ({df_features['is_female'].sum()/len(df_features)*100:.1f}%)")

# 4. Crear indicador de tabaquismo
print(f"\n[4] Creando indicador de tabaquismo...")
# SMQ020: 1=Sí ha fumado, 2=No ha fumado, 7=No sabe, 9=No responde
df_features['ever_smoked'] = (df_features['SMQ020'] == 1).astype(int)
print(f"  ✓ Ha fumado alguna vez: {df_features['ever_smoked'].sum()} ({df_features['ever_smoked'].sum()/len(df_features)*100:.1f}%)")

# 5. Verificar y reportar valores faltantes post-feature engineering
print(f"\n{'-'*70}")
print("RESUMEN DE NUEVAS CARACTERÍSTICAS:")
new_features = ['egfr', 'map', 'is_female', 'ever_smoked']
for feat in new_features:
    missing = df_features[feat].isna().sum()
    missing_pct = missing / len(df_features) * 100
    print(f"  - {feat}: {missing} faltantes ({missing_pct:.2f}%)")

print(f"\n✓ Feature engineering completado")



INGENIERÍA DE CARACTERÍSTICAS (FEATURE ENGINEERING)


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:9                                                                                    │
│                                                                                                  │
│    6 print("INGENIERÍA DE CARACTERÍSTICAS (FEATURE ENGINEERING)")                                │
│    7 print("="*70)                                                                               │
│    8                                                                                             │
│ ❱  9 df_features = df_merged.copy()                                                              │
│   10                                                                                             │
│   11 # 1. Calcular eGFR (CKD-EPI 2009)                                                           │
│   12 print(f"\n[1] Calculando eGFR (CKD-EPI 2009)...")                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'df_merged' is not defined

## 7. Ingeniería de Características (Feature Engineering)

### Cálculo de eGFR (Tasa de Filtración Glomerular Estimada)

La función de filtración renal es un predictor importante de mortalidad. Se estima mediante la **ecuación CKD-EPI 2009**, que ajusta por edad, género y creatinina sérica:

$$e\text{GFR} = 141 \times \min(S_{cr}/\kappa, 1)^{\alpha} \times \max(S_{cr}/\kappa, 1)^{-1.209} \times 0.993^{\text{age}}$$

Donde:
- $S_{cr}$ = creatinina sérica (mg/dL)
- $\kappa$ = 0.7 para mujeres, 0.9 para hombres
- $\alpha$ = -0.329 para mujeres, -0.411 para hombres

### Variables Derivadas Adicionales

- **BMI (IMC):** $\text{BMI} = \frac{\text{peso (kg)}}{\text{altura (m)^2}}$
- **MAP (Presión Arterial Media):** $\text{MAP} = \frac{P_{\text{sistólica}} + 2 \times P_{\text{diastólica}}}{3}$
- **Indicadores binarios:** Género, antecedente de tabaquismo

In [ ]:
# ============================================================================
# ANÁLISIS EXPLORATORIO DE DATOS (EDA)
# ============================================================================

print("\n" + "="*70)
print("ANÁLISIS EXPLORATORIO DE DATOS")
print("="*70)

# Información general
print(f"\nInformación del dataset:")
print(f"  - Forma: {df_merged.shape}")
print(f"  - Memoria: {df_merged.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"  - Tipos de datos:\n{df_merged.dtypes}")

# Valores faltantes
print(f"\n{'-'*70}")
print("VALORES FALTANTES:")
missing = df_merged.isnull().sum()
missing_pct = (missing / len(df_merged) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing_Count': missing[missing > 0],
    'Missing_%': missing_pct[missing > 0]
}).sort_values('Missing_%', ascending=False)

if len(missing_df) > 0:
    print(missing_df)
else:
    print("  [No hay valores faltantes]")

# Estadísticas descriptivas por variable
print(f"\n{'-'*70}")
print("ESTADÍSTICAS DESCRIPTIVAS:")
print(df_merged.describe(include='all').round(3))

# Análisis de la variable target (mortalidad)
print(f"\n{'-'*70}")
print("VARIABLE TARGET (MORTALIDAD):")
print(f"  - Eventos (mortstat=1): {(df_merged['mortstat'] == 1).sum()}")
print(f"  - No eventos (mortstat=0): {(df_merged['mortstat'] == 0).sum()}")
print(f"  - Tasa de mortalidad: {(df_merged['mortstat'] == 1).sum() / len(df_merged) * 100:.2f}%")

# Duración de seguimiento
print(f"\n  - Seguimiento (meses):")
print(f"    - Mínimo: {df_merged['permth_int'].min()}")
print(f"    - Máximo: {df_merged['permth_int'].max()}")
print(f"    - Mediana: {df_merged['permth_int'].median():.1f}")
print(f"    - Media: {df_merged['permth_int'].mean():.1f}")

print(f"\n✓ EDA inicial completada")


## 6. Análisis Exploratorio de Datos (EDA)

In [ ]:
# ============================================================================
# INTEGRACIÓN DE DATASETS
# ============================================================================

print("\n" + "="*70)
print("INTEGRACIÓN DE DATASETS")
print("="*70)

# Comenzar con mortalidad
df_merged = data_dict['mortality'].copy()
print(f"\n[BASE] Mortalidad: {df_merged.shape[0]} filas")

# Unir secuencialmente (INNER JOIN)
join_order = ['demo', 'biopro', 'bmx', 'bpx', 'smq']
for dataset_name in join_order:
    df_to_join = data_dict[dataset_name]
    df_before = df_merged.shape[0]
    
    df_merged = df_merged.merge(
        df_to_join,
        on='SEQN',
        how='inner',
        indicator=False
    )
    
    df_after = df_merged.shape[0]
    print(f"[+] {dataset_name}: {df_before} → {df_after} filas ({df_after - df_before:+d})")

print(f"\n{'-'*70}")
print(f"RESULTADO FINAL: {df_merged.shape[0]} filas × {df_merged.shape[1]} columnas")
print(f"\nColumnas disponibles:")
print(df_merged.columns.tolist())

# Guardar dataset integrado
output_path = INTERMEDIATE_DIR / 'merged_data.parquet'
df_merged.to_parquet(output_path, index=False)
print(f"\n✓ Dataset integrado guardado: {output_path}")


## 5. Integración de Datasets

Los datos se integran mediante la clave identificadora **SEQN** (Sequence Number), que es un identificador único para cada participante en NHANES.

**Estrategia de Merge:**
1. Comenzar con dataset de mortalidad (target)
2. Unir todos los datasets clínicos mediante INNER JOIN en SEQN
3. Conservar solo participantes con información completa

In [ ]:
# ============================================================================
# DESCARGAR DATOS
# ============================================================================

print("\n" + "="*70)
print("DESCARGA DE DATASETS DESDE CDC")
print("="*70)

# Mapeo de datasets
datasets = {
    'demo': {
        'url': CDC_URLS['DEMO'],
        'fallback': RAW_DATA_DIR / 'demo.parquet',
        'vars': DEMOGRAPHIC_VARS
    },
    'biopro': {
        'url': CDC_URLS['BIOPRO'],
        'fallback': RAW_DATA_DIR / 'biopro.parquet',
        'vars': LABORATORY_VARS
    },
    'bmx': {
        'url': CDC_URLS['BMX'],
        'fallback': RAW_DATA_DIR / 'bmx.parquet',
        'vars': ANTHROPOMETRY_VARS
    },
    'bpx': {
        'url': CDC_URLS['BPX'],
        'fallback': RAW_DATA_DIR / 'bpx.parquet',
        'vars': VITAL_SIGNS_VARS
    },
    'smq': {
        'url': CDC_URLS['SMQ'],
        'fallback': RAW_DATA_DIR / 'smq.parquet',
        'vars': SMOKING_VARS
    },
    'mortality': {
        'url': CDC_URLS['MORTALITY'],
        'fallback': RAW_DATA_DIR / 'mortality.parquet',
    }
}

# Descargar todos los datasets
data_dict = {}
for name, config in datasets.items():
    print(f"\n[{name.upper()}] ", end="")
    if name == 'mortality':
        df = load_mortality(config['url'], config['fallback'])
    else:
        df = download_xpt(config['url'], config['fallback'])
    
    data_dict[name] = df
    print(f"Forma: {df.shape}")

# Resumen de descargas
print(f"\n{'-'*70}")
print("RESUMEN DE DESCARGAS:")
for name, df in data_dict.items():
    print(f"  - {name}: {df.shape[0]} filas x {df.shape[1]} columnas")

print(f"\n✓ Descarga completada")


## 4. Descargar e Integrar Datasets

In [ ]:
# ============================================================================
# FUNCIONES DE INGESTA DE DATOS
# ============================================================================

def download_xpt(url: str, fallback_parquet: Path) -> pd.DataFrame:
    """
    Descargar archivo XPT desde CDC o cargar desde fallback parquet local.
    
    Args:
        url: URL del archivo XPT en CDC
        fallback_parquet: Ruta al archivo parquet local como fallback
    
    Returns:
        DataFrame con los datos
    """
    try:
        logging.info(f"Intentando descargar desde CDC: {url}")
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        # Descomprimir si es necesario
        content = response.content
        if url.endswith('.gz'):
            content = gzip.decompress(content)
        
        # Leer con pyreadstat
        df, meta = pyreadstat.read_xport(io.BytesIO(content))
        logging.info(f"[OK] Descargado exitosamente: {len(df)} registros")
        return df
    
    except Exception as e:
        logging.warning(f"Error descargando XPT ({type(e).__name__}): {str(e)[:50]}")
        
        # Fallback a parquet local
        if fallback_parquet.exists():
            logging.info(f"Cargando desde fallback: {fallback_parquet}")
            df = pd.read_parquet(fallback_parquet)
            logging.info(f"[OK] Cargado desde fallback: {len(df)} registros")
            return df
        else:
            logging.error(f"Fallback no disponible: {fallback_parquet}")
            return pd.DataFrame()

def load_mortality(url: str, fallback_parquet: Path) -> pd.DataFrame:
    """
    Descargar archivo DAT (mortalidad) desde CDC FTP o fallback.
    
    Args:
        url: URL FTP del archivo DAT
        fallback_parquet: Ruta al parquet local como fallback
    
    Returns:
        DataFrame con datos de mortalidad
    """
    try:
        logging.info(f"Intentando descargar mortalidad desde CDC: {url}")
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        # Parsear archivo DAT de ancho fijo
        content = response.text
        lines = content.strip().split('\n')
        
        # Especificaciones de ancho fijo (según documentación CDC)
        specs = [
            (0, 5),      # SEQN (5 dígitos)
            (5, 6),      # mortstat (1 dígito)
            (6, 15),     # permth_int (9 dígitos)
        ]
        
        data = []
        for line in lines:
            try:
                record = {}
                for col_name, (start, end) in zip(['SEQN', 'mortstat', 'permth_int'], specs):
                    record[col_name] = int(line[start:end])
                data.append(record)
            except:
                continue
        
        df = pd.DataFrame(data)
        logging.info(f"[OK] Descargado exitosamente: {len(df)} registros")
        return df
    
    except Exception as e:
        logging.warning(f"Error descargando mortalidad ({type(e).__name__}): {str(e)[:50]}")
        
        # Fallback a parquet local
        if fallback_parquet.exists():
            logging.info(f"Cargando desde fallback: {fallback_parquet}")
            df = pd.read_parquet(fallback_parquet)
            logging.info(f"[OK] Cargado desde fallback: {len(df)} registros")
            return df
        else:
            logging.error(f"Fallback no disponible: {fallback_parquet}")
            return pd.DataFrame()

print("✓ Funciones de ingesta definidas correctamente")


## 3. Funciones de Descarga e Ingesta de Datos

Las siguientes funciones descargan datasets desde CDC con fallback a archivos parquet locales para robustez y portabilidad.

In [ ]:
# ============================================================================
# DEFINICIÓN DE RUTAS Y PARÁMETROS
# ============================================================================

# Rutas de datos (relativas a raíz del proyecto)
DATA_DIR = project_root / 'data'
RAW_DATA_DIR = DATA_DIR / '01_raw' / 'nhanes_2013_2014'
INTERMEDIATE_DIR = DATA_DIR / '02_intermediate' / 'nhanes_2013_2014'
PRIMARY_DIR = DATA_DIR / '03_primary' / 'nhanes_2013_2014'

# Crear directorios si no existen
for directory in [RAW_DATA_DIR, INTERMEDIATE_DIR, PRIMARY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"\n[INFO] Rutas de datos configuradas:")
print(f"  - Raíz: {DATA_DIR}")
print(f"  - Raw: {RAW_DATA_DIR}")
print(f"  - Intermediate: {INTERMEDIATE_DIR}")
print(f"  - Primary: {PRIMARY_DIR}")

# Parámetros de configuración del análisis
# Variables a utilizar en el análisis
DEMOGRAPHIC_VARS = [
    'RIDAGEYR',  # Edad en años
    'RIAGENDR',  # Género (1=M, 2=F)
]

LABORATORY_VARS = [
    'LBXSCR',    # Creatinina sérica (mg/dL)
    'LBXBUN',    # Urea (BUN) (mg/dL)
]

VITAL_SIGNS_VARS = [
    'BPXSY1',    # Presión sistólica 1 (mmHg)
    'BPXDI1',    # Presión diastólica 1 (mmHg)
    'BPXPULS',   # Pulso (lpm)
]

ANTHROPOMETRY_VARS = [
    'BMXWT',     # Peso (kg)
    'BMXHT',     # Altura (cm)
    'BMXBMI',    # IMC (kg/m²)
]

SMOKING_VARS = [
    'SMQ020',    # Antecedente de tabaquismo
]

# URLs de datos CDC (con manejo de fallback)
CDC_URLS = {
    'DEMO': 'https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/DEMO_H.XPT',
    'BIOPRO': 'https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/BIOPRO_H.XPT',
    'BMX': 'https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/BMX_H.XPT',
    'BPX': 'https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/BPX_H.XPT',
    'SMQ': 'https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/SMQ_H.XPT',
    'MORTALITY': 'ftp://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/NHANES_2013_2014_MORT_2019_PUBLIC.dat',
}

# Variables de mortalidad (target)
DURATION_COL = 'permth_int'  # Meses desde seguimiento hasta evento
EVENT_COL = 'mortstat'       # 1=muerto, 0=vivo

print(f"\n[INFO] Variables seleccionadas para el análisis:")
print(f"  - Demográficas: {DEMOGRAPHIC_VARS}")
print(f"  - Laboratorio: {LABORATORY_VARS}")
print(f"  - Signos vitales: {VITAL_SIGNS_VARS}")
print(f"  - Antropometría: {ANTHROPOMETRY_VARS}")
print(f"  - Tabaquismo: {SMOKING_VARS}")


## 2. Rutas de Datos y Parámetros de Configuración

In [ ]:
# ============================================================================
# CONFIGURACIÓN INICIAL: RUTAS Y LIBRERÍAS
# ============================================================================

# Resolver rutas desde la subcarpeta notebooks/ (ir a carpeta padre)
import sys
from pathlib import Path

# Obtener ruta actual del notebook
notebook_dir = Path.cwd()
if notebook_dir.name != 'notebooks':
    # Si se ejecuta desde otra ubicación, ajustar manualmente
    notebook_dir = Path(__file__).parent if '__file__' in dir() else Path.cwd()

# Ir a la raíz del proyecto (padre del directorio notebooks/)
project_root = notebook_dir.parent if 'notebooks' in str(notebook_dir) else notebook_dir
sys.path.insert(0, str(project_root))

print(f"[INFO] Directorio del notebook: {notebook_dir}")
print(f"[INFO] Raíz del proyecto: {project_root}")
print(f"[INFO] Directorio de datos esperado: {project_root / 'data'}")

# Importar librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
import logging
from datetime import datetime
import requests
import gzip
import io
import pyreadstat

# Configuración visual
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Configuración de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Configuración de warnings
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')

# Reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("\n✓ Todas las librerías importadas correctamente")
print(f"✓ Semilla aleatoria establecida: {RANDOM_STATE}")


[INFO] Directorio del notebook: C:\Users\nholc\Desktop\ev3_nanhes\notebooks
[INFO] Raíz del proyecto: C:\Users\nholc\Desktop\ev3_nanhes
[INFO] Directorio de datos esperado: C:\Users\nholc\Desktop\ev3_nanhes\data


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:27                                                                                   │
│                                                                                                  │
│   24 import numpy as np                                                                          │
│   25 import pandas as pd                                                                         │
│   26 import matplotlib.pyplot as plt                                                             │
│ ❱ 27 import seaborn as sns                                                                       │
│   28 from scipy import stats                                                                     │
│   29 from sklearn.preprocessing import StandardScaler                                            │
│   30 from sklearn.model_selection import train_test_split                                        │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ModuleNotFoundError: No module named 'seaborn'

## 1. Importar Librerías y Configuración

# NHANES Mortality Prediction Pipeline
## Notebook 01: Ingesta de Datos y Análisis Exploratorio

**Objetivo:** Descargar, integrar y transformar datos de la cohorte NHANES 2013-2014 para análisis de mortalidad.

**Fuentes de Datos:**
- Encuesta Nacional de Examen de Salud y Nutrición (NHANES) 2013-2014
- Datos de mortalidad vinculados (CDC NCHS) hasta 2019

**Alcance de este Notebook:**
1. Descarga de 6 datasets (XPT + DAT)
2. Integración mediante claves identificadoras (SEQN)
3. Análisis Exploratorio de Datos (EDA) con estadísticas descriptivas y visualizaciones
4. Ingeniería de características (Feature Engineering): cálculo de eGFR, BMI, MAP, etc.
5. Limpieza, transformación y preparación final para modelado

**Requisitos Previos:**
- Python 3.10+
- Librerías: pandas, numpy, pyreadstat, requests, matplotlib, seaborn, scipy, scikit-learn
- Conexión a internet (para descarga de CDC) o archivos parquet locales como fallback

---